# 3장 예제 — 3.19 고차원 데이터와의 첫 대면 — SECOM (선택 학습, 20분)

- **교재 출처**: manuscript/31_ch3_eda_fe.md
- **실행 방법**: example 폴더에서  python "3장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 16_3-16_실전_정제_①_현장_엑셀_워크북_해체하기.py, 17_3-17_실전_정제_②_통계_규칙이_못_잡는_물리적_불가값.py, 18_3-18_실전_EDA.py
- **실행 상태**: 단독 실행 확인됨(이 파일만 돌려도 끝까지 실행된다).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

In [ ]:
# --- 실행 준비 (교재에서는 앞 절에서 이미 실행한 코드) --------------------
import pandas as pd

### 이 절에서 쓸 데이터 — 그리고 중요한 경고

In [ ]:
sec = pd.read_csv('data/secom/secom_merged.csv', parse_dates=['timestamp'])
sensors = [c for c in sec.columns if c.startswith('sensor_')]

print("모양:", sec.shape)
print("센서 컬럼 수:", len(sensors), " (문서 기재: 591)")
print(sec['label'].value_counts().to_string())
print("불량률:", round(100 * float(sec['fail'].mean()), 2), "%")

# 교재 실행 결과 ------------------------------------------------------
#   모양: (1567, 593)
#   센서 컬럼 수: 590  (문서 기재: 591)
#   label
#   -1    1463
#    1     104
#   불량률: 6.64 %

### dropna()가 데이터를 전멸시키는 순간

In [ ]:
row_na = sec[sensors].isna().sum(axis=1)
print("전체 결측률:", round(100 * float(sec[sensors].isna().mean().mean()), 2), "%")
print("결측이 0인 행:", int((row_na == 0).sum()), "개")
print("행별 결측 센서 수 — 중앙값:", int(row_na.median()), "· 최대:", int(row_na.max()))
print("dropna() 결과:", sec.dropna().shape)

# 교재 실행 결과 ------------------------------------------------------
#   전체 결측률: 4.54 %
#   결측이 0인 행: 0 개
#   행별 결측 센서 수 — 중앙값: 24 · 최대: 152
#   dropna() 결과: (0, 593)

### 열부터 정리한다 — 정보가 없는 센서 골라내기

In [ ]:
miss = sec[sensors].isna().mean()
nuniq = sec[sensors].nunique()

print("결측이 0인 컬럼:", int((miss == 0).sum()), "개")
print("결측 50% 초과 컬럼:", int((miss > 0.5).sum()), "개 (최대",
      round(100 * float(miss.max()), 2), "%)")
print("값이 한 종류뿐인(분산 0) 컬럼:", int((nuniq <= 1).sum()), "개 =",
      round(100 * float((nuniq <= 1).mean()), 1), "%")

drop = set(nuniq[nuniq <= 1].index) | set(miss[miss > 0.5].index)
keep = [c for c in sensors if c not in drop]
print("제거 대상 합집합:", len(drop), "개 → 남는 센서:", len(keep), "개")

# 교재 실행 결과 ------------------------------------------------------
#   결측이 0인 컬럼: 52 개
#   결측 50% 초과 컬럼: 28 개 (최대 91.19 %)
#   값이 한 종류뿐인(분산 0) 컬럼: 116 개 = 19.7 %
#   제거 대상 합집합: 144 개 → 남는 센서: 446 개

### 남은 446개도 그대로는 못 쓴다

In [ ]:
sd = sec[keep].std()
print("표준편차 최소:", round(float(sd.min()), 6), "· 최대:", round(float(sd.max()), 1))
print("최대/최소 비:", f"{float(sd.max() / sd.min()):,.0f}", "배")
print("---")
print("전부 정상으로 찍는 모델의 정확도:",
      round(100 * (1 - float(sec['fail'].mean())), 2), "%")
corr = sec[keep].corrwith(sec['fail']).abs().sort_values(ascending=False)
print("|corr(센서, 불량)| 상위 5개:")
print(corr.head(5).round(4).to_string())

# 교재 실행 결과 ------------------------------------------------------
#   표준편차 최소: 0.000104 · 최대: 6553.6
#   최대/최소 비: 62,768,858 배
#   ---
#   전부 정상으로 찍는 모델의 정확도: 93.36 %
#   |corr(센서, 불량)| 상위 5개:
#   sensor_059    0.1558
#   sensor_103    0.1512
#   sensor_510    0.1316
#   sensor_348    0.1302
#   sensor_431    0.1209

### 시간이 흐르면 공정도 변한다

In [ ]:
m = sec.groupby(sec['timestamp'].dt.to_period('M'))['fail'].agg(
    건수='size', 불량='sum', 불량률='mean')
m['불량률'] = (m['불량률'] * 100).round(2)
print(m.to_string())
print("중복 timestamp:", int(sec['timestamp'].duplicated().sum()), "건")

# 교재 실행 결과 ------------------------------------------------------
#               건수  불량    불량률
#   timestamp                
#   2008-07     63  14  22.22
#   2008-08    555  51   9.19
#   2008-09    590  17   2.88
#   2008-10    359  22   6.13
#   중복 timestamp: 33 건